# Day 9 — Tools, Function Calling & AI Agents

## Future Revision Notes

This notebook is based on the **original Day 9 learning notebook (`day4(1).ipynb`)**. The original learning cells are preserved at the end.

The separate practice notebook is treated only as evidence that you practiced; it is not used as the source of the lesson.

## What is a Tool?

A **tool** is a function that an LLM can request your application to execute.

The LLM does not directly execute your Python function.

```text
User
 ↓
LLM
 ↓
Tool call
 ↓
Your Python function
 ↓
Tool result
 ↓
LLM
 ↓
Final answer
```

For the airline example:

```python
get_ticket_price("Paris")
```

can return a ticket price.

### Most important idea

**The LLM decides/request a tool call. Your Python application executes the function.**

# 1. Normal LLM vs Tool-Using LLM

Without tools:

```text
User → LLM → Answer
```

With tools:

```text
User
 ↓
LLM
 ↓
Tool call
 ↓
Python function
 ↓
Tool result
 ↓
LLM
 ↓
Answer
```

Tools are useful for databases, calculations, APIs, flight searches, weather, email, records, retrieval, and actions.

This is one of the foundations of AI agents.

In [ ]:
# OpenAI setup — original lesson style

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
MODEL = "gpt-4.1-mini"

openai = OpenAI(api_key=openai_api_key)

In [ ]:
# Ollama setup — the local setup used in your practice

from openai import OpenAI

MODEL = "llama3.2:latest"

openai = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

# 2. System Prompt and Conversation History

The system prompt tells the airline assistant how to behave.

For a conversational chatbot, the application sends:

```text
system instruction
      ↓
previous history
      ↓
current user message
```

History is converted to the API message format and combined with the current message.

In [ ]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

def chat_without_tools(message, history):
    history = [
        {"role": h["role"], "content": h["content"]}
        for h in history
    ]

    messages = (
        [{"role": "system", "content": system_message}]
        + history
        + [{"role": "user", "content": message}]
    )

    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages
    )

    return response.choices[0].message.content

# 3. The Actual Python Tool

The original lesson starts with a simple ticket-price dictionary.

The Python function is the **actual implementation** of the tool.

```text
Tool schema
    ↓
Tells the LLM what the function does

Python function
    ↓
Actually performs the work
```

These are different things.

In [ ]:
ticket_prices = {
    "london": "$799",
    "paris": "$899",
    "tokyo": "$1400",
    "berlin": "$499"
}

def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")

    price = ticket_prices.get(
        destination_city.lower().strip(),
        "Unknown ticket price"
    )

    return f"The price of a ticket to {destination_city} is {price}"

print(get_ticket_price("London"))

# 4. Tool Schema

The LLM needs a structured description of the function.

The schema contains:

- function name
- description
- parameters
- parameter types
- parameter descriptions
- required parameters

The original lesson uses this OpenAI-style schema.

The schema **describes** the function; it does not execute it.

In [ ]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

tools = [
    {
        "type": "function",
        "function": price_function
    }
]

print(tools)

# 5. Complete Tool-Calling Flow

```text
1. User asks a question
        ↓
2. LLM receives question + tool definitions
        ↓
3. LLM decides whether a tool is needed
        ↓
4. LLM returns requested function + arguments
        ↓
5. Python reads the tool call
        ↓
6. Python executes the real function
        ↓
7. Python sends the result back to the LLM
        ↓
8. LLM produces the final answer
```

The model does **not** execute `get_ticket_price()` itself.

In [ ]:
def handle_tool_call(message):
    tool_call = message.tool_calls[0]

    if tool_call.function.name == "get_ticket_price":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get("destination_city")

        price_details = get_ticket_price(city)

        response = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }

    return response

In [ ]:
def chat_with_one_tool(message, history):
    history = [
        {"role": h["role"], "content": h["content"]}
        for h in history
    ]

    messages = (
        [{"role": "system", "content": system_message}]
        + history
        + [{"role": "user", "content": message}]
    )

    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools
    )

    if response.choices[0].finish_reason == "tool_calls":
        tool_message = response.choices[0].message
        tool_response = handle_tool_call(tool_message)

        messages.append(tool_message)
        messages.append(tool_response)

        response = openai.chat.completions.create(
            model=MODEL,
            messages=messages
        )

    return response.choices[0].message.content

# 6. Why `tool_call_id`?

A tool response contains:

```python
"tool_call_id": tool_call.id
```

This connects the returned tool result to the specific tool request.

```text
LLM requests tool
      ↓
tool_call_id
      ↓
Python executes function
      ↓
matching tool result
      ↓
LLM
```

This becomes especially important with multiple tool calls.

# 7. Multiple Tool Calls

The original lesson improves the implementation to handle more than one tool call.

Instead of:

```python
message.tool_calls[0]
```

use:

```python
for tool_call in message.tool_calls:
```

Then each call gets its own tool response.

In [ ]:
def handle_tool_calls(message):
    responses = []

    for tool_call in message.tool_calls:

        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get("destination_city")

            price_details = get_ticket_price(city)

            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })

    return responses

# 8. Multiple Rounds of Tool Calls

The original lesson then uses a `while` loop:

```python
while response.choices[0].finish_reason == "tool_calls":
```

Why?

The model may request another tool after receiving the first tool result.

```text
LLM
 ↓
Tool call
 ↓
Tool result
 ↓
LLM
 ↓
Another tool call?
 ├─ Yes → execute again
 └─ No  → final answer
```

This loop is a basic pattern behind tool-using AI agents.

In [ ]:
def chat_with_tools(message, history):
    history = [
        {"role": h["role"], "content": h["content"]}
        for h in history
    ]

    messages = (
        [{"role": "system", "content": system_message}]
        + history
        + [{"role": "user", "content": message}]
    )

    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools
    )

    while response.choices[0].finish_reason == "tool_calls":
        tool_message = response.choices[0].message
        responses = handle_tool_calls(tool_message)

        messages.append(tool_message)
        messages.extend(responses)

        response = openai.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )

    return response.choices[0].message.content

# 9. Database-Backed Tool

The original lesson replaces the dictionary with SQLite.

```text
LLM
 ↓
get_ticket_price(city)
 ↓
SQLite
 ↓
price
 ↓
LLM
```

This is more realistic because the tool retrieves data from a database rather than a hard-coded dictionary.

In [ ]:
import sqlite3

DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()

    cursor.execute(
        "CREATE TABLE IF NOT EXISTS prices "
        "(city TEXT PRIMARY KEY, price REAL)"
    )

    conn.commit()

In [ ]:
def get_ticket_price(city):
    print(
        f"DATABASE TOOL CALLED: Getting price for {city}",
        flush=True
    )

    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()

        cursor.execute(
            "SELECT price FROM prices WHERE city = ?",
            (city.lower().strip(),)
        )

        result = cursor.fetchone()

        return (
            f"Ticket price to {city} is ${result[0]}"
            if result
            else "No price data available for this city"
        )

In [ ]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()

        cursor.execute(
            """
            INSERT INTO prices (city, price)
            VALUES (?, ?)
            ON CONFLICT(city) DO UPDATE SET price = ?
            """,
            (city.lower().strip(), price, price)
        )

        conn.commit()


ticket_prices = {
    "london": 799,
    "paris": 899,
    "tokyo": 1420,
    "sydney": 2999
}

for city, price in ticket_prices.items():
    set_ticket_price(city, price)

print(get_ticket_price("Tokyo"))

# 10. Gemini — Same Concept, Different SDK

You asked to keep **both OpenAI and Gemini** versions for future revision.

The architecture is the same:

```text
OpenAI
  ↓
tool schema
  ↓
model requests function
  ↓
Python executes it
  ↓
tool result
  ↓
model

Gemini native SDK
  ↓
FunctionDeclaration / Tool
  ↓
model requests function
  ↓
Python executes it
  ↓
function response
  ↓
model
```

The SDK syntax differs, but the concept is the same.

In [ ]:
# Gemini native SDK — function declaration

from google import genai
from google.genai import types

gemini = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

GEMINI_MODEL = "gemini-3.5-flash-lite"

get_ticket_price_declaration = types.FunctionDeclaration(
    name="get_ticket_price",
    description="Get the price of a return ticket to the destination city.",
    parameters=types.Schema(
        type="OBJECT",
        properties={
            "destination_city": types.Schema(
                type="STRING",
                description="The city that the customer wants to travel to"
            )
        },
        required=["destination_city"]
    )
)

gemini_tool = types.Tool(
    function_declarations=[get_ticket_price_declaration]
)

In [ ]:
# Gemini native SDK — providing the tool to the model

def chat_gemini(message, history):
    contents = []

    for h in history:
        role = "user" if h["role"] == "user" else "model"

        contents.append(
            types.Content(
                role=role,
                parts=[types.Part.from_text(text=h["content"])]
            )
        )

    contents.append(
        types.Content(
            role="user",
            parts=[types.Part.from_text(text=message)]
        )
    )

    response = gemini.models.generate_content(
        model=GEMINI_MODEL,
        contents=contents,
        config=types.GenerateContentConfig(
            system_instruction=system_message,
            tools=[gemini_tool]
        )
    )

    return response

## Gemini native function-call handling

When Gemini requests a function, inspect the response parts for a function call.

The same conceptual steps apply:

```text
response
 ↓
find function call
 ↓
read function name
 ↓
read arguments
 ↓
execute Python function
 ↓
send function response back
 ↓
generate final answer
```

The exact response object structure is SDK-specific, so the installed Gemini SDK documentation should be used when implementing the complete loop.

# 11. Gemini Through the OpenAI-Compatible Endpoint

Another option is to use the OpenAI Python client against Gemini's OpenAI-compatible endpoint.

```text
OpenAI Python client
        ↓
Gemini OpenAI-compatible endpoint
        ↓
Gemini model
```

This is different from the native Gemini SDK:

```text
Native Gemini:
google.genai

OpenAI-compatible Gemini:
openai.OpenAI(base_url=Gemini endpoint)
```

The Chat Completions tool schema can then be reused.

In [ ]:
from openai import OpenAI

google_api_key = os.getenv("GEMINI_API_KEY")

gemini = OpenAI(
    api_key=google_api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

GEMINI_MODEL = "gemini-3.5-flash-lite"

def chat_gemini_compatible(message, history):
    history = [
        {"role": h["role"], "content": h["content"]}
        for h in history
    ]

    messages = (
        [{"role": "system", "content": system_message}]
        + history
        + [{"role": "user", "content": message}]
    )

    response = gemini.chat.completions.create(
        model=GEMINI_MODEL,
        messages=messages,
        tools=tools
    )

    return response

# 12. Greeting vs Tool Request

A tool is not required for every message.

For:

```text
Hi there
```

the model can answer directly.

For:

```text
What is the ticket price for Paris?
```

the model can choose:

```text
get_ticket_price("Paris")
```

So:

```text
Greeting → normal LLM response
Price question → tool call
```

# 13. Important Limitation of the Airline Tool

The original project has a **ticket-price lookup tool**.

It does not actually:

- search live flights
- check seats
- book flights
- check travel dates
- check origin airports
- process payments

Therefore:

```text
"How much is a ticket to Paris?"
```

fits the current tool.

But:

```text
"Book a flight from Los Angeles to New York next Wednesday."
```

would require a tool such as:

```python
search_flights(origin, destination, travel_date)
```

and potentially:

```python
book_flight(flight_id, passenger_details)
```

The LLM cannot perform those actions unless your application provides the appropriate tools/APIs.

# 14. Your Guntur Practice

Adding:

```python
"guntur": "$5000"
```

means the Python function can return `$5000`.

But:

```text
dictionary contains Guntur
```

does **not** guarantee:

```text
LLM will call the tool
```

The desired flow is:

```text
User: Check the price for Guntur
        ↓
LLM decides a tool is needed
        ↓
get_ticket_price("Guntur")
        ↓
"$5000"
        ↓
LLM receives tool result
        ↓
Final answer
```

A stronger system instruction can explicitly tell the model to use the tool for price questions.

In [ ]:
system_message = """
You are a helpful airline assistant.

When the user asks for a ticket price, cost, or fare,
use the get_ticket_price tool.

Do not invent ticket prices.

If the tool has no price data for the requested city,
clearly say that no price data is available.

For greetings such as "hi" or "hello",
respond normally without using the tool.
"""

# 15. Tools → AI Agents

Today's tool-calling loop is one of the foundations of an AI agent.

```text
                 LLM
                  ↓
           Decide next action
                  ↓
          ┌───────┴───────┐
          ↓               ↓
       Tool 1           Tool 2
          ↓               ↓
       Result            Result
          └───────┬───────┘
                  ↓
                 LLM
                  ↓
            Final answer
```

The model can understand the request, decide what action is needed, request a tool, receive the result, decide whether another tool is needed, and produce the final answer.

# Quick Recall Sheet

### Tool
A function that an LLM can request your application to execute.

### Tool schema
A structured description telling the model what the function does and what arguments it needs.

### Tool call
The model's structured request to execute a function.

### Tool result
The output of the Python function returned to the model.

### Core loop

```text
LLM decides
   ↓
Python executes
   ↓
Tool result
   ↓
LLM answers
```

### Multiple calls

```python
for tool_call in message.tool_calls:
    ...
```

### Multiple rounds

```python
while response.choices[0].finish_reason == "tool_calls":
    ...
```

### Database-backed tool

```text
LLM → Python tool → SQLite → result → LLM
```

# Day 9 Revision Exercise

Try these before looking back at the code.

### Exercise 1
Create:

```python
get_weather(city)
```

as a Python tool.

### Exercise 2
Create a tool schema for `get_weather(city)`.

### Exercise 3
Explain the difference between a tool schema and the actual Python function.

### Exercise 4
Why is the tool result sent back to the LLM?

### Exercise 5
Why can a `while` loop be better than a single `if` for tool calling?

### Exercise 6
Change:

```python
get_ticket_price(destination_city)
```

into:

```python
search_flights(origin, destination, travel_date)
```

What three required parameters would the schema need?

## Answers

**1.**
```python
def get_weather(city):
    return f"The weather in {city} is sunny."
```

**2.**
```python
weather_function = {
    "name": "get_weather",
    "description": "Get the weather for a city.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "The city to check"
            }
        },
        "required": ["city"],
        "additionalProperties": False
    }
}
```

**3.** The schema describes the function to the LLM. The Python function actually executes the operation.

**4.** The LLM needs the real tool result as context before it can produce the final answer.

**5.** The model may request another tool after receiving the first tool result.

**6.** The required parameters are `origin`, `destination`, and `travel_date`.

# Final Mental Model

If you remember only one thing from Day 9:

```text
User
 ↓
LLM decides
 ↓
Tool call
 ↓
Python executes
 ↓
Tool result
 ↓
LLM receives result
 ↓
Final answer
```

### Memory trick

> **LLM decides → Python executes → result goes back → LLM answers.**

# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()

# As an alternative, if you'd like to use Ollama instead of OpenAI
# Check that Ollama is running for you locally (see week1/day2 exercise) then uncomment these next 2 lines
# MODEL = "llama3.2"
# openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


In [ ]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

## Tools

Tools are an incredibly powerful feature provided by the frontier LLMs.

With tools, you can write a function, and have the LLM call that function as part of its response.

Sounds almost spooky.. we're giving it the power to run code on our machine?

Well, kinda.

In [ ]:
# Let's start by making a useful function

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    return f"The price of a ticket to {destination_city} is {price}"


In [ ]:
get_ticket_price("London")

In [ ]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [ ]:
# And this is included in a list of tools:

tools = [{"type": "function", "function": price_function}]

In [ ]:
tools

## Getting OpenAI to use our Tool

There's some fiddly stuff to allow OpenAI "to call our tool"

What we actually do is give the LLM the opportunity to inform us that it wants us to run the tool.

Here's how the new chat function looks:

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
 
    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [ ]:
# We have to write that function handle_tool_call:

def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_ticket_price":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city')
        price_details = get_ticket_price(city)
        response = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
    return response

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

## Let's make a couple of improvements

Handling multiple tool calls in 1 response

Handling multiple tool calls 1 after another

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [ ]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [ ]:
import sqlite3


In [ ]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [ ]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [ ]:
get_ticket_price("London")

In [ ]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [ ]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [ ]:
get_ticket_price("Tokyo")

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

## Exercise

Add a tool to set the price of a ticket!

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business Applications</h2>
            <span style="color:#181;">Hopefully this hardly needs to be stated! You now have the ability to give actions to your LLMs. This Airline Assistant can now do more than answer questions - it could interact with booking APIs to make bookings!</span>
        </td>
    </tr>
</table>